# Week 4: feature table

Owner: Abdulaziz.

This notebook builds `data/processed/features.csv` for the diabetes interviews. The split is the frozen Task 6 file, not a new draw: 60% train, 20% validation, 20% test, `random_state` 2024 (`SPLIT_SEED` in `src/diabetes_risk/config.py`). Drawing another split would make this week incomparable with the models already fit on those ids.

Shared cleaning stays in `src/diabetes_risk/clean.py`. This notebook calls `src/diabetes_risk/features.py` and writes down why each column exists.

The five steps stay in this order: split, encode, add domain columns, drop what does not help, scale only where a model needs it. Scaling is fit on the training rows and is not stored in the CSV, because the forest does not use it.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from diabetes_risk.config import SPLIT_SEED, TEST_FRAC, TRAIN_FRAC, VAL_FRAC
from diabetes_risk.features import (
    build_encoded,
    category_counts,
    load_prepared,
    modeling_checks,
    write_feature_table,
)

prepared, split = load_prepared()
print(split["split"].value_counts().reindex(["train", "validation", "test"]).to_string())
print(f"random_state={SPLIT_SEED}  train={TRAIN_FRAC}  validation={VAL_FRAC}  test={TEST_FRAC}")


split
train         24001
validation     8000
test           7999
random_state=2024  train=0.6  validation=0.2  test=0.2


## 1. Split first

The counts above are the frozen assignment, read before any median, mode, or encoding is computed. Validation and test rows are not allowed to move those statistics.


## 2. Encode the categories

Sex has two unordered answers, so it becomes one column, `sex_male`. Smoking, smokeless tobacco, and vaping have a real order (never, then former, then current use), so each becomes one ordinal column instead of a set of dummies. None of these columns has a long tail of levels, so nothing is collapsed into an "other" bucket. That rule is for a column with many rare levels, which this survey extract does not have.


In [2]:
counts = category_counts(prepared, split)
print(counts.assign(train_share=lambda frame: frame["train_share"].map(lambda value: f"{value:.1%}")).to_string(index=False))
widest = int(counts.groupby("column")["level"].nunique().max())
print(f"widest training column: {widest} levels")


           column             level  train_n train_share
              sex            female    12340       51.4%
              sex              male    11661       48.6%
    smoker_status             never    14500       60.4%
    smoker_status            former     6754       28.1%
    smoker_status current_every_day     1865        7.8%
    smoker_status current_some_days      739        3.1%
    smoker_status               NaN      143        0.6%
smokeless_tobacco        not_at_all    23342       97.3%
smokeless_tobacco         every_day      365        1.5%
smokeless_tobacco         some_days      248        1.0%
smokeless_tobacco               NaN       46        0.2%
       ecigarette             never    19267       80.3%
       ecigarette            former     3305       13.8%
       ecigarette         every_day      708        2.9%
       ecigarette         some_days      621        2.6%
       ecigarette               NaN      100        0.4%
     any_exercise              

## 3. Columns from the diabetes question

Each kept column has one line: how it is computed, and why it is here.

Age band uses the same cuts as the descriptive charts (18-29 through 70-80). BMI band uses the same WHO groups as those charts, because the diabetes class concentrates at BMI 30 and above. BMI divided by age is high when a high BMI shows up at a younger age: a BMI of 32 at 30 and a BMI of 32 at 75 are not the same profile.

Age band is built here and then judged in the next step. It is not kept just because it was built.


In [3]:
table, keys, report = build_encoded(prepared, split)
catalog = pd.DataFrame([
    {"feature": name, "how": how, "why": why}
    for name, (how, why) in report["notes"].items()
])
print(catalog.to_string(index=False))
for item in report["dropped_duplicates"]:
    print(
        f"built then dropped: {item['column']} "
        f"(correlation {item['correlation']:.3f} with {item['duplicate_of']})"
    )


       feature                                                                                      how                                                                                                     why
           age                                 Years from _AGE80. Missing ages use the training median.                                        People with diabetes are older (weighted mean 60.5 versus 47.1).
           bmi                              _BMI5 divided by 100. Missing BMI uses the training median.                                             BMI rises from 27.9 with no diabetes to 31.8 with diabetes.
  sugar_drinks                               SSBSUGR2 converted to times per day by the shared cleaner.                                                              Sugary soda is one of the contract inputs.
  any_exercise               EXERANY2 as 1 for yes and 0 for no. Missing answers use the training mode.                                            Exercise is lower wit

## 4. Drop what does not help

`row_id` is an identifier, so it is not in the table. Height and weight stay out: rebuilt BMI matches `_BMI5` with correlation 1.0, and the contract already keeps them descriptive-only. A column with one training value is dropped. If two training columns correlate at 0.98 or higher, the later one is dropped.

Age band correlates 0.984 with age on the training rows, so it is the column that goes. Age already carries that information, including the differences inside a band. BMI band correlates 0.941 with BMI, under the cutoff, and it stays because the clinical groups are not a copy of the continuous value. BMI divided by age also stays.


In [4]:
print("constants dropped:", report["dropped_constants"] or "none")
print("near-duplicates dropped:", report["dropped_duplicates"] or "none")
features = [column for column in table.columns if column not in ("target", "split")]
focus = [column for column in ("age", "age_band", "bmi", "bmi_band", "bmi_to_age") if column in features]
train = table.loc[table["split"].eq(0), features]
print(train[focus].corr().round(3).to_string())
print("row_id in the modeling table:", "row_id" in table.columns)


constants dropped: none
near-duplicates dropped: [{'column': 'age_band', 'duplicate_of': 'age', 'correlation': 0.9843691410884292}]
              age    bmi  bmi_band  bmi_to_age
age         1.000 -0.021    -0.017      -0.825
bmi        -0.021  1.000     0.941       0.442
bmi_band   -0.017  0.941     1.000       0.408
bmi_to_age -0.825  0.442     0.408       1.000
row_id in the modeling table: False


## 5. Scale only for the model that needs it, and only from the training rows

The CSV stays in the original units. Logistic regression scales `age`, `bmi`, `sugar_drinks`, and `bmi_to_age` inside its pipeline, with `StandardScaler.fit` on the training rows. The forest is not scaled. The means below are a check: after a train-only fit, the training means are 0 and the validation means are not forced to 0.


In [5]:
continuous = [column for column in ("age", "bmi", "sugar_drinks", "bmi_to_age") if column in features]
scaler = StandardScaler().fit(train[continuous])
valid = table.loc[table["split"].eq(1), continuous]
print("training BMI median used for the fill:", round(report["train_medians"]["bmi"], 4))
print("all-rows BMI median before that fill:", round(report["all_row_medians_before_impute"]["bmi"], 4))
print("train means after scaling:", scaler.transform(train[continuous]).mean(axis=0).round(6))
print("validation means with that same scaler:", scaler.transform(valid).mean(axis=0).round(3))


training BMI median used for the fill: 27.46
all-rows BMI median before that fill: 27.44


train means after scaling: [ 0. -0. -0.  0.]
validation means with that same scaler: [ 0.014 -0.011 -0.004 -0.021]


## Checks before the file is handed on

Four questions: are the columns numeric, are the missing values gone, is the identifier gone, and do the training and test column counts match? `target` is 0 no diabetes, 1 prediabetes, 2 diabetes. `split` is 0 train, 1 validation, 2 test. Neither column is a model input. `feature_keys.csv` keeps `row_id` beside this file so later predictions can be joined; it is not fed to a model.


In [6]:
paths = write_feature_table(table, keys)
checks = modeling_checks(table)
for name, value in checks.items():
    print(f"{name}: {value}")
print("wrote", paths[0].name, "and", paths[1].name)
if not (checks["all_numeric"] and checks["no_missing"] and checks["identifier_removed"] and checks["same_column_count"]):
    raise RuntimeError(checks)


all_numeric: True
no_missing: True
identifier_removed: True
train_columns: 11
test_columns: 11
same_column_count: True
same_column_names: True
train_rows: 24001
validation_rows: 8000
test_rows: 7999
wrote features.csv and feature_keys.csv


## Who did this work

This week: Abdulaziz (this notebook and `data/processed/features.csv`).

Names already recorded for earlier weeks, included here because the handout asks for them with this submission:

- Abdulaziz: project layout, analysis contract, week-3 exploratory notebook
- Latifah: thin table and column dictionary
- Nasser Abanmy: cleaning functions and the frozen split
- Abdulmalk Alnajem: data-quality profile and relationship charts
- Bandar Alshahrani: diabetes-type table
- Latifah Alsulihem: training, metrics, and the first model comparison
